## Putting Everything Together: The Full Query Lifecycle

Query rewriting and decomposition are pre-retrieval transformations. They happen after routing, since you need to know where you are searching before you can rewrite optimally, and before the actual vector search.

The full query lifecycle through our Enterprise RAG system now follows this sequence:

1. **Raw query arrives** at the pipeline entry point.
2. **Semantic cache check** -- if a semantically similar query has been answered before, return the cached result immediately. No further processing needed.
3. **Time-sensitivity filter** -- if the query contains temporal keywords, bypass the cache and route directly to live web search.
4. **Route query** -- the LLM-based router classifies intent and selects the target collection.
5. **Rewrite query** -- the rewriter expands abbreviations, resolves references, and adds domain specificity.
6. **Decompose if compound** -- if the rewritten query contains multiple information needs, split it into atomic sub-queries.
7. **Retrieve from target collection** -- each atomic query (or the single rewritten query) searches the routed Qdrant collection or SerpApi.
8. **Synthesize and return response** -- GPT-4o generates a grounded, cited answer from the retrieved chunks.
9. **Cache the result** -- the query-response pair is stored in the semantic cache for future reuse.

![Full query lifecycle pipeline](../assets/ch07__image006.png)

In [ ]:
async def enterprise_rag_pipeline(
    user_query: str,
    cache: SemanticCaching,
    conversation_history: list = None
) -> dict:
    result = {
        "query": user_query,
        "rewritten_query": None,
        "sub_queries": None,
        "route": None,
        "reason": None,
        "cache_hit": False,
        "time_sensitive": False,
        "answer": None,
    }
 
    # Step 1: Check semantic cache
    hit, cached_answer, embedding, sim, _ = cache.check_cache(
        user_query
    )
    if hit:
        result["cache_hit"] = True
        result["answer"] = cached_answer
        return result
 
    # Step 2: Check time sensitivity
    if is_time_sensitive(user_query):
        result["time_sensitive"] = True
        search_results = search_web(user_query)
        result["answer"] = rag_formatted_response(
            user_query, search_results
        )
        return result
 
    # Step 3: Route the query
    route_result = route_query(user_query)
    result["route"] = route_result["action"]
    result["reason"] = route_result["reason"]
 
    if route_result.get("answer"):
        result["answer"] = route_result["answer"]
        cache.add_to_cache(user_query, result["answer"],
                           embedding)
        return result
 
    # Step 4: Rewrite the query
    rewritten = rewrite_query(
        user_query, conversation_history
    )
    result["rewritten_query"] = rewritten
 
    # Step 5: Decompose if compound
    sub_queries = decompose_query(rewritten)
    result["sub_queries"] = sub_queries
 
    # Step 6: Retrieve and generate for each sub-query
    action = route_result["action"]
    if action == "WEB_SEARCH":
        all_context = []
        for sq in sub_queries:
            all_context.extend(search_web(sq))
    else:
        all_context = []
        for sq in sub_queries:
            sq_embedding = get_text_embeddings(sq)
            collections = {
                "OPENAI_QUERY": "opnai_data",
                "10K_DOCUMENT_QUERY": "10k_data",
            }
            hits = await qdrant.query_points(
                collection_name=collections[action],
                query=sq_embedding,
                limit=3
            )
            all_context.extend(
                [p.payload["content"] for p in hits.points]
            )
 
    # Step 7: Synthesize final answer
    result["answer"] = rag_formatted_response(
        user_query, all_context
    )
 
    # Step 8: Cache the result
    cache.add_to_cache(
        user_query, result["answer"], embedding
    )
 
    return result